In [ ]:
import pandas as pd
import re


# ============================================================
# 1. CAMINHOS DOS ARQUIVOS
# ============================================================

arquivo_dados = "04_PENSE2024_SEM_NAN.parquet"

arquivo_regras = "01_Colunas_recomendadas_pense_2024_final.xlsx"

arquivo_saida = "05_PENSE2024_NUMERACAO_NOVA.parquet"


# ============================================================
# 2. LER OS ARQUIVOS
# ============================================================

df = pd.read_parquet(arquivo_dados)

regras = pd.read_excel(arquivo_regras)


# ============================================================
# 3. IDENTIFICAR AS COLUNAS IMPORTANTES DO EXCEL
# ============================================================

coluna_codigo = "Código da coluna"
coluna_numeracao = "Nova numeracao"


# Verificar se as colunas existem
if coluna_codigo not in regras.columns:
    raise ValueError(
        f"A coluna '{coluna_codigo}' não foi encontrada no Excel."
    )

if coluna_numeracao not in regras.columns:
    raise ValueError(
        f"A coluna '{coluna_numeracao}' não foi encontrada no Excel."
    )


# ============================================================
# 4. FUNÇÃO PARA INTERPRETAR A REGRA
# ============================================================

def criar_mapa(regra):
    """
    Converte uma regra como:

        1=4,2=2,3=3,4=1,-9/-2/-1=0

    em um dicionário:

        {
            "1": "4",
            "2": "2",
            "3": "3",
            "4": "1",
            "-9": "0",
            "-2": "0",
            "-1": "0"
        }
    """

    mapa = {}

    if pd.isna(regra):
        return mapa

    regra = str(regra).strip()

    # Separar cada regra pela vírgula
    partes = regra.split(",")

    for parte in partes:

        parte = parte.strip()

        if not parte:
            continue

        # Procurar o "="
        if "=" not in parte:
            continue

        esquerda, direita = parte.split("=", 1)

        esquerda = esquerda.strip()
        direita = direita.strip()

        # Pode haver vários valores do lado esquerdo:
        #
        # -9/-2/-1=0
        #
        # Nesse caso todos recebem 0.
        valores_origem = esquerda.split("/")

        for valor in valores_origem:

            valor = valor.strip()

            if valor:
                mapa[valor] = direita

    return mapa


# ============================================================
# 5. PROCESSAR AS COLUNAS
# ============================================================

total_colunas_processadas = 0
total_valores_alterados = 0

relatorio = []


for _, linha in regras.iterrows():

    codigo = linha[coluna_codigo]
    regra = linha[coluna_numeracao]

    # Ignorar linhas sem código
    if pd.isna(codigo):
        continue

    codigo = str(codigo).strip()

    # Ignorar linhas sem regra
    if pd.isna(regra):
        continue

    regra = str(regra).strip()

    if not regra:
        continue

    # Verificar se a coluna existe no Parquet
    if codigo not in df.columns:

        print(f"ATENÇÃO: coluna '{codigo}' não encontrada no Parquet.")

        continue

    # Criar o mapa de conversão
    mapa = criar_mapa(regra)

    if not mapa:
        print(f"ATENÇÃO: regra inválida para '{codigo}': {regra}")
        continue

    # Garantir que os valores sejam strings
    df[codigo] = df[codigo].astype("string")

    # Guardar valores antes da transformação
    valores_antes = df[codigo].copy()

    # Fazer a substituição
    df[codigo] = df[codigo].replace(mapa)

    # Contar quantos valores foram modificados
    alterados = (valores_antes != df[codigo]).sum()

    total_valores_alterados += alterados
    total_colunas_processadas += 1

    # Guardar relatório
    relatorio.append({
        "coluna": codigo,
        "regra": regra,
        "valores_alterados": alterados
    })

    print(
        f"{codigo:15s} | "
        f"Alterados: {alterados:8,d} | "
        f"Regra: {regra}"
    )


# ============================================================
# 6. SALVAR O NOVO ARQUIVO
# ============================================================

df.to_parquet(
    arquivo_saida,
    index=False
)


# ============================================================
# 7. RELATÓRIO FINAL
# ============================================================

relatorio_df = pd.DataFrame(relatorio)

print("\n" + "=" * 70)
print("PROCESSAMENTO CONCLUÍDO")
print("=" * 70)

print(f"Colunas processadas:       {total_colunas_processadas}")
print(f"Valores alterados:         {total_valores_alterados:,}")
print(f"Arquivo original:          {arquivo_dados}")
print(f"Arquivo novo:              {arquivo_saida}")

print("\nRelatório:")
print(relatorio_df.to_string(index=False))

Colunas encontradas na planilha:
'Table 1'
'Unnamed: 1'
'Unnamed: 2'
'Unnamed: 3'
'Unnamed: 4'
'Unnamed: 5'
'Unnamed: 6'
'Unnamed: 7'
'Unnamed: 8'
'Unnamed: 9'
'Unnamed: 10'
'Unnamed: 11'
'Unnamed: 12'
'Unnamed: 13'


In [ ]:
import pandas as pd
import unicodedata

# Ler a planilha
df_ref = pd.read_excel(
    "01_Colunas_recomendadas_pense_2024_final.xlsx"
)

# Função para normalizar nomes das colunas
def normalizar_nome(nome):

    nome = str(nome).strip().lower()

    nome = unicodedata.normalize(
        "NFKD", nome
    ).encode("ASCII", "ignore").decode("ASCII")

    nome = " ".join(nome.split())

    return nome

# Criar correspondência entre nomes normalizados e originais
colunas_normalizadas = {
    normalizar_nome(coluna): coluna
    for coluna in df_ref.columns
}

# Localizar as colunas necessárias
col_codigo = colunas_normalizadas["codigo da coluna"]

col_observacao = colunas_normalizadas[
    "observacoes do dicionario"
]

# Padronizar os nomes para o restante do código
df_ref = df_ref.rename(
    columns={
        col_codigo: "Código da coluna",
        col_observacao: "Observações do dicionário"
    }
)

# Limpar os códigos
df_ref["Código da coluna"] = (
    df_ref["Código da coluna"]
    .astype(str)
    .str.strip()
)

print("Colunas identificadas corretamente!")

print("Código:", "Código da coluna")
print("Regra:", "Observações do dicionário")

In [1]:

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import re
from pathlib import Path

# ==========================================
# 1. Arquivos
# ==========================================

arquivo_parquet = Path(
    "05_PENSE2024_SEM_NAN_SEM_DUPLICADOS.parquet"
)

arquivo_referencia = Path(
    "01_Colunas_recomendadas_pense_2024_final.xlsx"
)

arquivo_saida = Path(
    "06_PENSE2024_RECODIFICADO.parquet"
)

arquivo_relatorio = Path(
    "06_RELATORIO_RECODIFICACAO.xlsx"
)

# ==========================================
# 2. Ler a planilha de referência
# ==========================================

df_ref = pd.read_excel(arquivo_referencia)

df_ref["Código da coluna"] = (
    df_ref["Código da coluna"]
    .astype(str)
    .str.strip()
)

# ==========================================
# 3. Interpretar as regras
# ==========================================

def interpretar_regra(texto):

    if pd.isna(texto):
        return {}

    texto = str(texto)

    # Identificar expressões do tipo:
    # 1/2=1
    # 3=2
    # -9/-2/-1=0

    padrao = r"([-\d\s/]+)\s*=\s*(-?\d+)"

    correspondencias = re.findall(
        padrao,
        texto
    )

    mapa = {}

    for valores, destino in correspondencias:

        valores_origem = re.findall(
            r"-?\d+",
            valores
        )

        for valor in valores_origem:

            mapa[int(valor)] = int(destino)

    return mapa


# ==========================================
# 4. Criar dicionário de recodificação
# ==========================================

regras = {}

for _, linha in df_ref.iterrows():

    codigo = linha["Código da coluna"]

    observacao = linha["Observações do dicionário"]

    mapa = interpretar_regra(observacao)

    if mapa:
        regras[codigo] = mapa


# ==========================================
# 5. Verificar colunas disponíveis
# ==========================================

parquet = pq.ParquetFile(arquivo_parquet)

colunas_base = parquet.schema.names

regras_aplicaveis = {
    codigo: mapa
    for codigo, mapa in regras.items()
    if codigo in colunas_base
}

print("Variáveis com regras:", len(regras))
print("Variáveis encontradas:", len(regras_aplicaveis))


# ==========================================
# 6. Processar e recodificar
# ==========================================

escritor = None
relatorio = []

total_linhas = 0

for lote in pq.ParquetFile(
    arquivo_parquet
).iter_batches(batch_size=10000):

    bloco = lote.to_pandas()

    for coluna, mapa in regras_aplicaveis.items():

        valores_antes = bloco[coluna].copy()

        # Aplicar somente os valores definidos no mapa
        bloco[coluna] = (
            bloco[coluna]
            .map(mapa)
            .fillna(valores_antes)
        )

    tabela = pa.Table.from_pandas(
        bloco,
        preserve_index=False
    )

    if escritor is None:

        escritor = pq.ParquetWriter(
            arquivo_saida,
            tabela.schema,
            compression="snappy"
        )

    escritor.write_table(tabela)

    total_linhas += len(bloco)

    print(
        f"Linhas processadas: {total_linhas:,}"
    )

if escritor is not None:
    escritor.close()


# ==========================================
# 7. Criar relatório das regras
# ==========================================

for codigo, mapa in regras_aplicaveis.items():

    relatorio.append({
        "Código da coluna": codigo,
        "Regra aplicada": str(mapa)
    })

df_relatorio = pd.DataFrame(relatorio)

df_relatorio.to_excel(
    arquivo_relatorio,
    index=False
)


# ==========================================
# 8. Resultado
# ==========================================

print("\nPROCESSAMENTO CONCLUÍDO")

print("Total de linhas:", total_linhas)

print("Variáveis recodificadas:", len(regras_aplicaveis))

print("Arquivo Parquet:", arquivo_saida)

print("Relatório:", arquivo_relatorio)


KeyError: 'Código da coluna'

In [2]:
import pandas as pd
from IPython.display import display, Markdown

# 1. Carregar o dicionário de dados
dicionario = pd.read_excel("00_Dicionário_Microdados_PENSE2024.xlsx")

dico_map = {}
for _, row in dicionario.iterrows():
    desc = str(row["Descrição da Variável"]).strip()
    cat = str(row["Categoria"]) if pd.notna(row["Categoria"]) else ""
    cat_dict = {}
    if cat:
        for linha in cat.split("\n"):
            if " - " in linha:
                partes = linha.split(" - ", 1)
                cat_dict[partes[0].strip()] = partes[1].strip()
    dico_map[desc] = cat_dict

# 2. Gerar a tabela em Markdown
total = len(df_limpo)
markdown_lines = [
    "| Pergunta / Variável | Distribuição das Respostas |",
    "| :--- | :--- |"
]

for col in df_limpo.columns:
    counts = df_limpo[col].astype(str).str.strip().value_counts(dropna=False)
    cat_dict = dico_map.get(col, {})
    
    respostas_formatadas = []
    for cod, qtd in counts.items():
        rotulo = cat_dict.get(cod, "")
        pct = (qtd / total) * 100
        if rotulo:
            respostas_formatadas.append(f"[{cod}] {rotulo}: {pct:.2f}%")
        else:
            respostas_formatadas.append(f"[{cod}]: {pct:.2f}%")
            
    # Subsitui quebras de linha normais por <br> do HTML para a tabela Markdown respeitar a formatação
    distribuicao_str = "<br>".join(respostas_formatadas)
    
    # Limpa quebras de linha no nome da pergunta para não quebrar a tabela Markdown
    col_clean = str(col).replace("\n", " ").replace("|", "-")
    
    markdown_lines.append(f"| {col_clean} | {distribuicao_str} |")

# 3. Exibir a tabela renderizada
print("========== EXIBIÇÃO DOS RESULTADOS NO IPYNB ==========\n")
display(Markdown("\n".join(markdown_lines)))

========== EXIBIÇÃO DOS RESULTADOS NO IPYNB ==========



| Pergunta / Variável | Distribuição das Respostas |
| :--- | :--- |
| NOS ÚLTIMOS 12 MESES, alguma vez você sentiu vontade de se machucar de propósito? | [2] Não: 67.16%<br>[1] Sim: 32.84% |
| NOS ÚLTIMOS 30 DIAS, com que frequência você sentiu que a vida não vale a pena ser vivida? | [1] Nunca: 49.17%<br>[2] Raramente: 17.57%<br>[3] Às vezes: 15.70%<br>[4] Na maioria das vezes: 8.85%<br>[5] Sempre: 8.71% |
| NOS ÚLTIMOS 30 DIAS, com que frequência você se sentiu muito preocupado com as coisas comuns do seu dia a dia como atividades da escola, competições esportivas, tarefas de casa, etc.? | [4] Na maioria das vezes: 28.25%<br>[3] Às vezes: 25.71%<br>[5] Sempre: 25.41%<br>[2] Raramente: 11.70%<br>[1] Nunca: 8.93% |
| NOS ÚLTIMOS 30 DIAS, com que frequência você sentiu que ninguém se preocupa com você? | [1] Nunca: 27.73%<br>[3] Às vezes: 24.56%<br>[2] Raramente: 22.52%<br>[4] Na maioria das vezes: 14.35%<br>[5] Sempre: 10.85% |
| NOS ÚLTIMOS 30 DIAS, com que frequência você se sentiu irritado(a), nervoso(a) ou mal-humorado(a) por qualquer coisa? | [3] Às vezes: 32.20%<br>[4] Na maioria das vezes: 25.22%<br>[5] Sempre: 17.89%<br>[2] Raramente: 15.81%<br>[1] Nunca: 8.87% |
| Quantos(as) amigos(as) próximos você tem? | [4] 3 ou mais amigos: 76.16%<br>[3] 2 amigos: 13.79%<br>[2] 1 amigo: 6.36%<br>[1] Nenhum amigo: 3.70% |
| NOS ÚLTIMOS 30 DIAS, com que frequência os colegas de sua escola trataram você bem e/ou foram prestativos com você? | [4] Na maioria das vezes: 38.47%<br>[5] Sempre: 27.64%<br>[3] Às vezes: 18.58%<br>[2] Raramente: 9.08%<br>[1] Nunca: 6.23% |
| NOS ÚLTIMOS 30 DIAS, quantas vezes algum dos seus colegas de escola o esculachou, zoou, mangou, intimidou ou caçoou tanto que você ficou magoado, incomodado, aborrecido, ofendido ou humilhado? | [1] Nenhuma vez nos últimos 30 dias: 58.70%<br>[4] 3 ou mais vezes: 19.11%<br>[2] 1 vez: 13.20%<br>[3] 2 vezes: 8.99% |
| NOS ÚLTIMOS 30 DIAS, quantas vezes algum dos seus colegas de escola se recusou a falar com você, deixou você de lado sem razão ou fez com que outros colegas deixassem de falar com você? | [1] Nenhuma vez nos últimos 30 dias: 72.99%<br>[2] 1 vez: 11.94%<br>[4] 3 ou mais vezes: 9.23%<br>[3] 2 vezes: 5.84% |
| NOS ÚLTIMOS 30 DIAS, quantas vezes algum dos seus colegas de escola bateu (deu socos, tapas, chutes, pontapés) em você ou o machucou fisicamente de outra forma? | [1] Nenhuma vez nos últimos 30 dias: 82.05%<br>[2] 1 vez: 7.64%<br>[4] 3 ou mais vezes: 6.57%<br>[3] 2 vezes: 3.74% |
| NOS ÚLTIMOS 30 DIAS, você se sentiu ameaçado(a), ofendido(a) ou humilhado(a) nas redes sociais ou aplicativos de celular? | [2] Não: 87.81%<br>[1] Sim: 12.19% |
| NOS ÚLTIMOS 30 DIAS, você esculachou, zombou, mangou, intimidou ou caçoou algum(a) de seus(suas) colegas da escola tanto que ele(ela) ficou magoado(a), aborrecido(a), ofendido(a) ou humilhado(a)? | [2] Não: 86.66%<br>[1] Sim: 13.34% |
| NOS ÚLTIMOS 30 DIAS, você ameaçou, ofendeu ou humilhou outras pessoas nas redes sociais ou aplicativos de celular? | [2] Não: 89.61%<br>[1] Sim: 10.39% |
| NOS ÚLTIMOS 12 MESES, quantas vezes você foi agredido(a) fisicamente por sua mãe, pai ou responsável? | [1] Nenhuma vez nos últimos 12 meses: 78.69%<br>[2] 1 vez: 10.10%<br>[3] 2 a 5 vezes: 7.49%<br>[4] 6 ou mais vezes: 3.73% |
| NOS ÚLTIMOS 12 MESES, quantas vezes você foi agredido(a) fisicamente por OUTRA PESSOA que não seja sua mãe, pai ou responsável? | [1] Nenhuma vez nos últimos 12 meses: 85.74%<br>[2] 1 vez: 7.50%<br>[3] 2 a 5 vezes: 4.47%<br>[4] 6 ou mais vezes: 2.29% |
| NOS ÚLTIMOS 30 DIAS, em quantos dias você deixou de ir à escola porque não se sentia seguro NO CAMINHO de casa para a escola ou da escola para casa? | [1] Nenhum dia nos últimos 30 dias: 89.93%<br>[2] 1 dia: 3.77%<br>[3] 2 dias: 2.27%<br>[6] 5 dias ou mais: 1.93%<br>[4] 3 dias: 1.37%<br>[5] 4 dias: 0.72% |
| NOS ÚLTIMOS 30 DIAS, em quantos dias você não foi à escola porque não se sentia seguro NA ESCOLA? | [1] Nenhum dia nos últimos 30 dias: 88.19%<br>[2] 1 dia: 4.35%<br>[3] 2 dias: 2.59%<br>[6] 5 dias ou mais: 2.38%<br>[4] 3 dias: 1.64%<br>[5] 4 dias: 0.85% |
| NOS ÚLTIMOS 30 DIAS, com que frequência sua mãe, pai ou responsável sabia realmente o que você estava fazendo em seu tempo livre? | [5] Sempre: 34.21%<br>[4] Na maioria das vezes: 34.08%<br>[3] Às vezes: 15.29%<br>[2] Raramente: 8.84%<br>[1] Nunca: 7.58% |
| NOS ÚLTIMOS 30 DIAS, com que frequência sua mãe, pai ou responsável entendeu suas preocupações e problemas? | [5] Sempre: 23.06%<br>[4] Na maioria das vezes: 22.07%<br>[3] Às vezes: 20.40%<br>[2] Raramente: 17.42%<br>[1] Nunca: 17.05% |
| NOS ÚLTIMOS 30 DIAS, com que frequência seus pais ou responsáveis mexeram em suas coisas sem a sua concordância? | [1] Nunca: 42.09%<br>[2] Raramente: 26.36%<br>[3] Às vezes: 17.28%<br>[4] Na maioria das vezes: 8.14%<br>[5] Sempre: 6.13% |
| Você mora com sua mãe? | [1] Sim: 90.11%<br>[2] Não: 9.89% |
| Você mora com seu pai? | [1] Sim: 63.68%<br>[2] Não: 36.32% |
| Qual é o seu sexo? | [2] Mulher: 50.53%<br>[1] Homem: 49.47% |
| IDADE DO ESTUDANTE, CATEGORIAS | [2] 13 a 15 anos: 51.68%<br>[3] 16 e 17 anos: 27.85%<br>[1] Menos de 13 anos: 16.03%<br>[4] 18 anos ou mais: 4.44% |
| Qual é a sua cor ou raça? | [4] Parda: 43.43%<br>[1] Branca: 41.16%<br>[2] Preta: 10.47%<br>[3] Amarela: 2.73%<br>[5] Indígena: 2.20% |
| código de regiões | [2] nordeste: 33.63%<br>[1] norte: 24.17%<br>[3] sudeste: 18.41%<br>[5] centro-oeste: 12.32%<br>[4] sul: 11.46% |
| código UF | [32] Espírito Santo: 4.92%<br>[35] São Paulo: 4.85%<br>[26] Pernambuco: 4.53%<br>[29] Bahia: 4.53%<br>[33] Rio de Janeiro: 4.38%<br>[31] Minas Gerais: 4.27%<br>[11] Rondônia: 4.25%<br>[41] Paraná: 4.16%<br>[42] Santa Catarina: 4.07%<br>[17] Tocantins: 4.00%<br>[50] Mato Grosso do Sul: 3.99%<br>[21] Maranhão: 3.93%<br>[28] Sergipe: 3.87%<br>[27] Alagoas: 3.83%<br>[13] Amazonas: 3.60%<br>[25] Paraíba: 3.51%<br>[15] Pará: 3.51%<br>[22] Piauí: 3.33%<br>[43] Rio Grande do Sul: 3.23%<br>[12] Acre: 3.16%<br>[16] Amapá: 3.12%<br>[23] Ceará: 3.07%<br>[24] Rio Grande do Norte: 3.05%<br>[51] Mato Grosso: 2.96%<br>[52] Goiás: 2.88%<br>[14] Roraima: 2.53%<br>[53] Distrito Federal: 2.49% |
| indica se é município da capital | [1] é capital: 51.12%<br>[0] não é capital: 48.88% |
| Na sua casa tem computador ou notebook? | [1] Sim: 63.81%<br>[2] Não: 36.19% |
| Você tem acesso à internet em sua casa? | [1] Sim: 97.44%<br>[2] Não: 2.56% |
| Quantas horas por dia você costuma ficar sentado(a), assistindo televisão, jogando videogame, usando computador, celular, tablet ou fazendo outras atividades sentado(a)? (NÃO contar sábado, domingo, feriados ou o tempo sentado na escola) | [1] Até 1 hora por dia: 14.18%<br>[3] Mais de 2 horas até 3 horas por dia: 14.11%<br>[2] Mais de 1 hora até 2 horas por dia: 13.70%<br>[4] Mais de 3 horas até 4 horas por dia: 12.51%<br>[5] Mais de 4 horas até 5 horas por dia: 9.96%<br>[10] Não sei / não lembro: 9.84%<br>[9] Mais de 8 horas por dia: 8.85%<br>[6] Mais de 5 horas até 6 horas por dia: 7.77%<br>[7] Mais de 6 horas até 7 horas por dia: 5.17%<br>[8] Mais de 7 horas até 8 horas por dia: 3.92% |
| NOS ÚLTIMOS 7 DIAS, sem contar as aulas de educação física da escola, em quantos dias você praticou alguma atividade física? | [1] Nenhum dia nos últimos 7 dias: 28.95%<br>[3] 2 dias: 13.10%<br>[4] 3 dias: 12.43%<br>[6] 5 dias: 12.22%<br>[2] 1 dia: 10.56%<br>[8] 5 dias mais sábado e domingo: 9.40%<br>[5] 4 dias: 8.09%<br>[7] 5 dias mais sábado: 5.25% |
| NOS ÚLTIMOS 12 MESES, a escola contou com algum suporte psicológico para situações de bullying e violências ocorridas na escola para alunos, professores e funcionários? | [1] Sim, para alunos, professores e funcionários: 51.41%<br>[2] Sim, somente para alunos: 28.88%<br>[4] Não: 18.32%<br>[3] Sim, para professores e funcionários: 1.40% |
| NOS ÚLTIMOS 12 MESES, a escola desenvolveu quais dessas seguintes ações? - Prevenção de violências e acidentes | [1] Sim: 56.20%<br>[0] Não: 43.80% |
| NOS ÚLTIMOS 12 MESES, a escola desenvolveu quais dessas seguintes ações? - Prevenção de práticas de bullying nas dependências da escola | [1] Sim: 51.44%<br>[0] Não: 48.56% |